# Tarea 3
**Integrantes**: Cristóbal Navarro y Cristopher Vasquez
### Instrucciones de uso
1. Tener la carpeta "BD" descomprimida en la misma carpeta donde se encuentra este ipynb.
2. Ejecutar celdas en orden.
### Instalación de librerías
Primero instalaremos todas las librerías necesarias para ejecutar el programa:

In [ ]:
%pip install pandas opencv-python scikit-image

## Función para calculo de descriptores por canal
Hacemos una función capaz de sacar:
- Contraste
- Energia
- ASM
- Homogeneidad
- Correlación
- Disimilaridad

A esta función le enviaremos cada imagen con su respectivo canal RGB y en escala de grises.

In [ ]:
from skimage.feature import graycomatrix, graycoprops

def obtencionCaracteristicas (img):
    glcm = graycomatrix(img, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    contraste = graycoprops(glcm, prop="contrast")[0,0]
    energia = graycoprops(glcm, prop= "energy")[0,0]
    ASM = graycoprops(glcm, prop = "ASM")[0,0]
    homogeneidad = graycoprops(glcm, prop= "homogeneity")[0,0]
    correlacion = graycoprops(glcm, prop= "correlation")[0,0]
    disimilaridad = graycoprops(glcm, prop= "dissimilarity")[0,0]
    
    return contraste, energia, ASM, homogeneidad, correlacion, disimilaridad 


### Extracción de Descriptores
Ya hecho esto definimos el path de la base de datos como `BD` y calculamos, para cada imagen, sus respectivos descriptores.
La imagen es separada por canal mediante la función `cv2.split()` y obtenemos el canal en escala de grises mediante `cv2.cvtColor(...,cv2.COLOR_BGR2GRAY)`.

In [11]:
import cv2
import pandas as pd
from pathlib import Path
from collections import defaultdict

CARPETA = Path("BD")
Path("resultados").mkdir(exist_ok=True)

contador = defaultdict(int)   # {clase: n_imágenes_vistas}
filas = []

for ruta in sorted(CARPETA.rglob("*.jpg")):
    img = cv2.imread(str(ruta))
    if img is None:
        print(f"⚠️  No se pudo leer: {ruta}")
        continue

    clase = ruta.parent.name
    contador[clase] += 1
    img_id = f"{clase}_img{contador[clase]:02d}"   # textura_01_img01

    azul, verde, rojo = cv2.split(img)
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    fila = {"id": img_id, "clase": clase}

    for nombre, canal in zip(("azul", "verde", "rojo", "gris"), (azul, verde, rojo, gris)):
        contraste, energia, ASM, homogeneidad, correlacion, disimilaridad = \
            obtencionCaracteristicas(canal)

        fila[f"{nombre}_contraste"]     = contraste
        fila[f"{nombre}_energia"]       = energia
        fila[f"{nombre}_ASM"]           = ASM
        fila[f"{nombre}_homogeneidad"]  = homogeneidad
        fila[f"{nombre}_correlacion"]   = correlacion
        fila[f"{nombre}_disimilaridad"] = disimilaridad

    filas.append(fila)

df = pd.DataFrame(filas)

print(df[["id", "clase"]].head(10))
print(df.shape)
print(df["clase"].value_counts())

df.to_csv("resultados/Caracteristicas.csv", index=False)
print("\n✅ Guardado en Caracteristicas.csv")

                 id       clase
0  textura_01_img01  textura_01
1  textura_01_img02  textura_01
2  textura_01_img03  textura_01
3  textura_01_img04  textura_01
4  textura_01_img05  textura_01
5  textura_01_img06  textura_01
6  textura_01_img07  textura_01
7  textura_01_img08  textura_01
8  textura_01_img09  textura_01
9  textura_01_img10  textura_01
(500, 26)
clase
textura_01    50
textura_02    50
textura_03    50
textura_04    50
textura_05    50
textura_06    50
textura_07    50
textura_08    50
textura_09    50
textura_10    50
Name: count, dtype: int64

✅ Guardado en Caracteristicas.csv


### Normalización de Características
Ahora, con las características obtenidas, normalizaremos cada una de las clases para obtener $\mu = 0$ y $\sigma = 1$.

In [12]:
import pandas as pd

df = pd.read_csv("resultados/Caracteristicas.csv")

# Separar las 24 características de los identificadores y etiquetas.
X = df.drop(columns=["id", "clase"])

# Calcular los parámetros por columna.
medias = X.mean()
desviaciones = X.std(ddof=0)

# Normalizar. La sustitución evita dividir por cero en columnas constantes.
X_normalizado = (X - medias) / desviaciones.replace(0, 1)

# Verificar lo que pide la rúbrica.
verificacion = pd.DataFrame({
    "media": X_normalizado.mean(),
    "desviacion": X_normalizado.std(ddof=0)
})

display(verificacion.round(6))

# Conservar identificadores y clases sin modificarlos.
df_normalizado = df.copy()
df_normalizado[X.columns] = X_normalizado

df_normalizado.to_csv("resultados/Caracteristicas_normalizadas.csv", index=False)

,media,desviacion
azul_contraste,-0.0,1.0
azul_energia,0.0,1.0
azul_ASM,-0.0,1.0
azul_homogeneidad,0.0,1.0
azul_correlacion,-0.0,1.0
azul_disimilaridad,-0.0,1.0
verde_contraste,-0.0,1.0
verde_energia,0.0,1.0
verde_ASM,-0.0,1.0
verde_homogeneidad,-0.0,1.0


## Separación del conjunto

aqui separai la wea en conjuntos de validacion y haces la parte mas bacan de to el trabajo, >:(

### Referencias

1. https://scikit--image-org.translate.goog/docs/stable/auto_examples/features_detection/plot_glcm.html?_x_tr_sl=en&_x_tr_tl=es&_x_tr_hl=es&_x_tr_pto=sge

2. https://scikit--image-org.translate.goog/docs/stable/api/skimage.feature.html?_x_tr_sl=en&_x_tr_tl=es&_x_tr_hl=es&_x_tr_pto=tc#skimage.feature.graycoprops